In [1]:
# --- Configuration ---
HL7_DIR = r"C:\Data Generator\output"   # folder containing .hl7 or .txt files
OUTPUT_DIR = r"C:\Data Generator\output\parsed"  # where CSVs will be written

# --- Imports ---
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Using:", HL7_DIR)



Using: C:\Data Generator\output


In [2]:
# --- HL7 helpers (Spark UDF equivalents in plain Python) ---

SEGMENT_REGEX = {
    'MSH': re.compile(r'(?s)(^|\r|\n)(MSH\|[^\r\n]*)'),
    'PID': re.compile(r'(?s)(^|\r|\n)(PID\|[^\r\n]*)'),
    'PV1': re.compile(r'(?s)(^|\r|\n)(PV1\|[^\r\n]*)'),
    'OBR': re.compile(r'(?s)(^|\r|\n)(OBR\|[^\r\n]*)'),
    'ORC': re.compile(r'(?s)(^|\r|\n)(ORC\|[^\r\n]*)'),
    'FT1': re.compile(r'(?s)(^|\r|\n)(FT1\|[^\r\n]*)'),
}

def get_segment(raw_message: str, seg: str):
    if raw_message is None or seg not in SEGMENT_REGEX:
        return None
    m = SEGMENT_REGEX[seg].search(raw_message)
    return m.group(2) if m else None

def field(raw_segment: str, idx: int):
    # HL7 fields are 0-based in this function: parts[0] = "SEG"
    if raw_segment is None:
        return None
    parts = raw_segment.split('|')
    return parts[idx] if 0 <= idx < len(parts) else None

def component(raw_field: str, comp_idx: int):
    # 1-based component index to mimic Spark version in your notebook
    if raw_field is None:
        return None
    comps = raw_field.split('^')
    return comps[comp_idx-1] if 1 <= comp_idx <= len(comps) else None

def trim_nullish(s: str):
    if s is None:
        return None
    s = s.strip()
    return s if s and s not in {'', 'null', 'NULL', '""'} else None

def normalize_id(s: str):
    if s is None:
        return None
    s = s.upper().strip()
    s = re.sub(r'[\s\-]', '', s)
    return s or None

def to_iso_date_from_ts(ts: str):
    # HL7 TS like YYYYMMDDHHMMSS... → YYYY-MM-DD
    if not ts:
        return None
    m = re.match(r'^(\d{4})(\d{2})(\d{2})', ts)
    if not m:
        return None
    return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"


In [3]:
def read_all_messages_from_dir(dir_path: str):
    """
    Reads every .hl7/.txt file in the directory and splits into messages.
    We consider a new message to start whenever a line begins with 'MSH|'.
    """
    dirp = Path(dir_path)
    files = sorted([*dirp.glob("*.hl7"), *dirp.glob("*.txt")])
    messages = []
    for fp in files:
        with fp.open("r", encoding="utf-8", errors="ignore") as f:
            buf = []
            for line in f:
                line = line.rstrip("\n\r")
                if line.startswith("MSH|"):
                    if buf:
                        messages.append("\r".join(buf))
                        buf = []
                buf.append(line)
            if buf:
                messages.append("\r".join(buf))
    return messages

all_messages = read_all_messages_from_dir(HL7_DIR)
print(f"Loaded {len(all_messages)} HL7 messages from {HL7_DIR}")


Loaded 5381 HL7 messages from C:\Data Generator\output


In [4]:
def parse_identifiers(msg: str):
    """
    Extracts the same identifier set you used:
    - message_control_id (MSH-10)
    - message_datetime (MSH-7) and date
    - message_type (MSH-9)
    - patient_ids_norm from PID-3 (repetitions by ~, take component 1 as ID)
    - account_number (PID-18)
    - visit_number (PV1-19)
    - filler_order_number (OBR-3)
    - placer_order_number (ORC-2)
    - ft1_proc_code_raw (FT1-19)  [DFT only typically]
    """
    seg_msh = get_segment(msg, "MSH")
    seg_pid = get_segment(msg, "PID")
    seg_pv1 = get_segment(msg, "PV1")
    seg_obr = get_segment(msg, "OBR")
    seg_orc = get_segment(msg, "ORC")
    seg_ft1 = get_segment(msg, "FT1")

    message_control_id = trim_nullish(field(seg_msh, 9))
    message_datetime   = trim_nullish(field(seg_msh, 6))
    message_date       = to_iso_date_from_ts(message_datetime)
    message_type       = trim_nullish(field(seg_msh, 8))

    pid3_list = trim_nullish(field(seg_pid, 3))
    patient_ids = pid3_list.split("~") if pid3_list else []
    # normalize: take component 1 (ID), upper/trim, remove spaces/dashes
    patient_ids_norm = []
    for rep in patient_ids:
        _id = component(rep, 1)
        if _id is not None:
            _id = normalize_id(_id)
            if _id:
                patient_ids_norm.append(_id)
    patient_ids_norm = sorted(set(patient_ids_norm)) if patient_ids_norm else []

    account_number = normalize_id(trim_nullish(field(seg_pid, 18)))
    visit_number   = normalize_id(trim_nullish(field(seg_pv1, 19)))
    filler_order_number = normalize_id(trim_nullish(field(seg_obr, 3)))
    placer_order_number = normalize_id(trim_nullish(field(seg_orc, 2)))
    ft1_proc_code_raw   = trim_nullish(field(seg_ft1, 19))

    return {
        "message_hash": msg,  # use raw message as hash like the Spark job
        "message_control_id": message_control_id,
        "message_datetime": message_datetime,
        "message_date": message_date,
        "message_type": message_type,
        "patient_ids_norm": patient_ids_norm,
        "account_number": account_number,
        "visit_number": visit_number,
        "location": trim_nullish(field(seg_pv1, 3)),
        "filler_order_number": filler_order_number,
        "placer_order_number": placer_order_number,
        "ft1_proc_code_raw": ft1_proc_code_raw
    }

def classify_source(msg_type: str):
    # rough mapping: startswith ORU^ → ORU, DFT^ → DFT, else Unknown
    if not msg_type:
        return "Unknown"
    mt = msg_type.upper()
    if mt.startswith("ORU^"):
        return "ORU"
    if mt.startswith("DFT^"):
        return "DFT"
    if mt.startswith("ADT^"):
        return "ADT"
    return "Unknown"

parsed = []
for m in all_messages:
    ident = parse_identifiers(m)
    src = classify_source(ident["message_type"])
    ident["source"] = src
    parsed.append(ident)

df_all = pd.DataFrame(parsed)
print("Parsed:", len(df_all))
df_all.head(3)


Parsed: 5381


,message_hash,message_control_id,message_datetime,message_date,message_type,patient_ids_norm,account_number,visit_number,location,filler_order_number,placer_order_number,ft1_proc_code_raw,source
0,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,5847c56f-1d62-40a4-a2d1-aa3caa884dfd,20250823142702,2025-08-23,ADT^A01^ADT_A01,[RAD2530753],414110417,VN7991355271,RAD_DEPT1,None,None,None,ADT
1,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,5836bd89-684e-4777-a6dd-852a778b6c1a,20250823142703,2025-08-23,ADT^A01^ADT_A01,[RAD8053401],139317086,VN3060029846,RAD_DEPT1,None,None,None,ADT
2,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,51780c56-e53f-4b8b-afcf-3577d3d4e1e5,20250823142703,2025-08-23,ADT^A01^ADT_A01,[RAD7636026],759656730,VN2417233836,RAD_DEPT1,None,None,None,ADT


In [5]:
def explode_keys(df: pd.DataFrame, include_filler: bool):
    """
    Create rows with columns:
      key_type ∈ {MRN, AccountNumber, VisitNumber, PlacerOrderNumber, [FillerOrderNumber]}
      key_value
      + basic message metadata
    Deduplicate by (message_hash, key_type, key_value).
    Always returns a DataFrame with expected columns, even if empty.
    """
    expected_cols = [
        "message_hash","message_control_id","message_date","message_datetime",
        "message_type","source","key_type","key_value"
    ]
    if df is None or df.empty:
        return pd.DataFrame(columns=expected_cols)

    rows = []
    base_cols = ["message_hash","message_control_id","message_date","message_datetime","message_type","source"]

    def add_row(rec, key_type, key_value):
        if key_value:
            rows.append({
                **{c: rec.get(c) for c in base_cols},
                "key_type": key_type,
                "key_value": key_value
            })

    for rec in df.to_dict("records"):
        for mrn in rec.get("patient_ids_norm") or []:
            add_row(rec, "MRN", mrn)
        add_row(rec, "AccountNumber", rec.get("account_number"))
        add_row(rec, "VisitNumber",   rec.get("visit_number"))
        add_row(rec, "PlacerOrderNumber", rec.get("placer_order_number"))
        if include_filler:
            add_row(rec, "FillerOrderNumber", rec.get("filler_order_number"))

    keys = pd.DataFrame(rows, columns=expected_cols)
    if keys.empty:
        return keys

    keys = keys.dropna(subset=["key_value"]).drop_duplicates(
        subset=["message_hash","key_type","key_value"]
    )
    return keys

df_oru = df_all[df_all["source"] == "ORU"].copy()
df_dft = df_all[df_all["source"] == "DFT"].copy()

oru_keys = explode_keys(df_oru, include_filler=True)
dft_keys = explode_keys(df_dft, include_filler=False)

print("ORU msgs:", len(df_oru), "ORU keys:", len(oru_keys))
print("DFT msgs:", len(df_dft), "DFT keys:", len(dft_keys))
print("oru_keys columns:", list(oru_keys.columns))
print("dft_keys columns:", list(dft_keys.columns))


ORU msgs: 2128 ORU keys: 6594
DFT msgs: 1206 DFT keys: 2622
oru_keys columns: ['message_hash', 'message_control_id', 'message_date', 'message_datetime', 'message_type', 'source', 'key_type', 'key_value']
dft_keys columns: ['message_hash', 'message_control_id', 'message_date', 'message_datetime', 'message_type', 'source', 'key_type', 'key_value']


In [6]:
# Inner join on (key_type, key_value), but guard for empties
expected_match_cols = [
    "key_type","key_value",
    "oru_message_hash","oru_msh10","oru_msg_date","oru_msg_datetime","oru_msg_type",
    "dft_message_hash","dft_msh10","dft_msg_date","dft_msg_datetime","dft_msg_type"
]

if oru_keys.empty or dft_keys.empty:
    match_df = pd.DataFrame(columns=expected_match_cols)
else:
    match_cols = ["key_type","key_value"]
    m = oru_keys.merge(
        dft_keys,
        on=match_cols,
        how="inner",
        suffixes=("_oru","_dft")
    )

    match_df = m.loc[:, [
        "key_type","key_value",
        "message_hash_oru","message_control_id_oru","message_date_oru","message_datetime_oru","message_type_oru",
        "message_hash_dft","message_control_id_dft","message_date_dft","message_datetime_dft","message_type_dft"
    ]].rename(columns={
        "message_hash_oru": "oru_message_hash",
        "message_control_id_oru": "oru_msh10",
        "message_date_oru": "oru_msg_date",
        "message_datetime_oru": "oru_msg_datetime",
        "message_type_oru": "oru_msg_type",
        "message_hash_dft": "dft_message_hash",
        "message_control_id_dft": "dft_msh10",
        "message_date_dft": "dft_msg_date",
        "message_datetime_dft": "dft_msg_datetime",
        "message_type_dft": "dft_msg_type",
    })

    match_df = match_df.drop_duplicates(
        subset=["key_type","key_value","oru_message_hash","dft_message_hash"]
    )

print("Matched pairs:", len(match_df))

matched_oru_msgs = match_df["oru_message_hash"].nunique() if not match_df.empty else 0
matched_dft_msgs = match_df["dft_message_hash"].nunique() if not match_df.empty else 0
total_oru = len(df_oru)
total_dft = len(df_dft)

stats = {
    "total_oru": total_oru,
    "matched_oru": matched_oru_msgs,
    "pct_oru_matched": round(matched_oru_msgs / total_oru, 4) if total_oru else 0.0,
    "total_dft": total_dft,
    "matched_dft": matched_dft_msgs,
    "pct_dft_matched": round(matched_dft_msgs / total_dft, 4) if total_dft else 0.0
}
stats


Matched pairs: 4466


{'total_oru': 2128,
 'matched_oru': 2128,
 'pct_oru_matched': 1.0,
 'total_dft': 1206,
 'matched_dft': 1200,
 'pct_dft_matched': 0.995}

In [7]:
# Full parsed identifiers for ORU/DFT
df_oru.to_csv(Path(OUTPUT_DIR, "hl7_oru_identifiers.csv"), index=False)
df_dft.to_csv(Path(OUTPUT_DIR, "hl7_dft_identifiers.csv"), index=False)

# Keys and matches
oru_keys.to_csv(Path(OUTPUT_DIR, "oru_keys.csv"), index=False)
dft_keys.to_csv(Path(OUTPUT_DIR, "dft_keys.csv"), index=False)
match_df.to_csv(Path(OUTPUT_DIR, "oru_dft_matches.csv"), index=False)

print("Wrote:")
print(Path(OUTPUT_DIR, "hl7_oru_identifiers.csv"))
print(Path(OUTPUT_DIR, "hl7_dft_identifiers.csv"))
print(Path(OUTPUT_DIR, "oru_keys.csv"))
print(Path(OUTPUT_DIR, "dft_keys.csv"))
print(Path(OUTPUT_DIR, "oru_dft_matches.csv"))


Wrote:
C:\Data Generator\output\parsed\hl7_oru_identifiers.csv
C:\Data Generator\output\parsed\hl7_dft_identifiers.csv
C:\Data Generator\output\parsed\oru_keys.csv
C:\Data Generator\output\parsed\dft_keys.csv
C:\Data Generator\output\parsed\oru_dft_matches.csv


In [8]:
print("Counts of non-null identifier fields (ORU):")
print(df_oru[["patient_ids_norm","account_number","visit_number","placer_order_number","filler_order_number"]].notna().sum())

print("\nCounts of non-null identifier fields (DFT):")
print(df_dft[["patient_ids_norm","account_number","visit_number","placer_order_number"]].notna().sum())

print("\nSample ORU rows with any identifier present:")
display(df_oru[(df_oru["patient_ids_norm"].str.len()>0) |
               df_oru["account_number"].notna() |
               df_oru["visit_number"].notna() |
               df_oru["placer_order_number"].notna() |
               df_oru["filler_order_number"].notna()].head(5))

print("\nSample DFT rows with any identifier present:")
display(df_dft[(df_dft["patient_ids_norm"].str.len()>0) |
               df_dft["account_number"].notna() |
               df_dft["visit_number"].notna() |
               df_dft["placer_order_number"].notna()].head(5))


Counts of non-null identifier fields (ORU):
patient_ids_norm       2128
account_number          210
visit_number           2128
placer_order_number       0
filler_order_number    2128
dtype: int64

Counts of non-null identifier fields (DFT):
patient_ids_norm       1206
account_number          210
visit_number           1206
placer_order_number       0
dtype: int64

Sample ORU rows with any identifier present:


,message_hash,message_control_id,message_datetime,message_date,message_type,patient_ids_norm,account_number,visit_number,location,filler_order_number,placer_order_number,ft1_proc_code_raw,source
3253,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,184c5fc8-0a18-41ad-a921-78b050e7b375,20250823171334,2025-08-23,ORU^R01^ORU_R01,[RAD8811028],716373398,VN1860126483,RAD_DEPT1,C7B5795D652B428D9B075C01C7620D2E,None,None,ORU
3254,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,924f98f8-b589-4f75-a2e0-50ba5853dcad,20250823171335,2025-08-23,ORU^R01^ORU_R01,[RAD6545770],784298067,VN9008443806,RAD_DEPT1,2B4E7A4448F84CEE8C275835E4114C7A,None,None,ORU
3255,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,765ae213-1f9c-4b2c-a629-d50e384d6b9f,20250823171335,2025-08-23,ORU^R01^ORU_R01,[RAD3087886],521834793,VN6335990196,RAD_DEPT1,B5E6DE53E5644DB68B639A36D2DA24B5,None,None,ORU
3256,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,4486c28c-4477-4e08-aed8-61246c8f1a7f,20250823171336,2025-08-23,ORU^R01^ORU_R01,[RAD3026154],337357246,VN2866036521,RAD_DEPT1,FAA94A0B5B95487F90303FC3862D9985,None,None,ORU
3257,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,11b739c9-757a-40c9-b374-4b2f40ff9d7d,20250823171336,2025-08-23,ORU^R01^ORU_R01,[RAD4912312],280217081,VN7486662477,RAD_DEPT1,C28971D84D9944748B59C499802EF437,None,None,ORU



Sample DFT rows with any identifier present:


,message_hash,message_control_id,message_datetime,message_date,message_type,patient_ids_norm,account_number,visit_number,location,filler_order_number,placer_order_number,ft1_proc_code_raw,source
1114,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,0571913c-2041-4bdb-9853-1d484501b210,20250823171334,2025-08-23,DFT^P03^DFT_P03,[RAD8811028],716373398,VN1860126483,RAD_DEPT1,C7B5795D652B428D9B075C01C7620D2E,None,None,DFT
1115,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,710f7009-0c6c-4caf-aa46-1b857c130896,20250823171335,2025-08-23,DFT^P03^DFT_P03,[RAD6545770],784298067,VN9008443806,RAD_DEPT1,2B4E7A4448F84CEE8C275835E4114C7A,None,None,DFT
1116,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,34e0bd3f-fa3f-4e8a-8e77-99045b25bd98,20250823171335,2025-08-23,DFT^P03^DFT_P03,[RAD3087886],521834793,VN6335990196,RAD_DEPT1,B5E6DE53E5644DB68B639A36D2DA24B5,None,None,DFT
1117,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,658d4872-0864-41a1-a941-37cd583f8566,20250823171336,2025-08-23,DFT^P03^DFT_P03,[RAD3026154],337357246,VN2866036521,RAD_DEPT1,FAA94A0B5B95487F90303FC3862D9985,None,None,DFT
1118,MSH|^~\&|FAKELAB|FAKEFACILITY|CMX|STAGE|202508...,2a18bccc-17ec-4323-a1c0-8dab6fc3fceb,20250823171336,2025-08-23,DFT^P03^DFT_P03,[RAD4912312],280217081,VN7486662477,RAD_DEPT1,C28971D84D9944748B59C499802EF437,None,None,DFT


In [9]:
# If you don't have duckdb yet (run once):
# !pip install duckdb --quiet

import duckdb, json
from pathlib import Path

DUCKDB_PATH = r"C:\Data Generator\output\hl7.duckdb"   # change if you like
con = duckdb.connect(DUCKDB_PATH)
print("Connected to", DUCKDB_PATH)


Connected to C:\Data Generator\output\hl7.duckdb


In [10]:
def with_json_lists(df, list_cols):
    if df is None or df.empty:
        return df
    df2 = df.copy()
    for c in list_cols:
        if c in df2.columns:
            df2[c] = df2[c].apply(lambda x: json.dumps(x) if isinstance(x, (list, tuple)) else (json.dumps([]) if x is None else json.dumps([x]) if not isinstance(x, str) and pd.notna(x) else json.dumps([])))
            # rename to make type explicit
            df2.rename(columns={c: f"{c}_json"}, inplace=True)
    return df2

oru_for_db = with_json_lists(df_oru, ["patient_ids_norm"])
dft_for_db = with_json_lists(df_dft, ["patient_ids_norm"])
# keys/matches have no list columns; use as-is
oru_keys_for_db = oru_keys.copy()
dft_keys_for_db = dft_keys.copy()
matches_for_db  = match_df.copy()


In [11]:
# Helper to replace a table from a pandas DF
def save_table(name, df):
    if df is None:
        raise ValueError(f"{name} is None")
    # DuckDB: register then CREATE OR REPLACE TABLE ... AS SELECT ...
    con.register("df_tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM df_tmp")
    con.unregister("df_tmp")
    rows = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"Wrote {rows:,} rows to {name}")

save_table("hl7_oru_identifiers", oru_for_db)
save_table("hl7_dft_identifiers", dft_for_db)
save_table("hl7_oru_keys", oru_keys_for_db)
save_table("hl7_dft_keys", dft_keys_for_db)
save_table("hl7_oru_dft_matches", matches_for_db)


Wrote 2,128 rows to hl7_oru_identifiers
Wrote 1,206 rows to hl7_dft_identifiers
Wrote 6,594 rows to hl7_oru_keys
Wrote 2,622 rows to hl7_dft_keys
Wrote 4,466 rows to hl7_oru_dft_matches


In [12]:
print(con.execute("SELECT COUNT(*) FROM hl7_oru_identifiers").fetchall())
print(con.execute("SELECT key_type, COUNT(*) FROM hl7_oru_dft_matches GROUP BY 1 ORDER BY 2 DESC").fetchdf().head(10))


[(2128,)]
        key_type  count_star()
0            MRN          2128
1    VisitNumber          2128
2  AccountNumber           210


In [13]:
import os
from pathlib import Path

SNAPSHOT_PATH = r"C:\Data Generator\data\hl7.duckdb"
os.makedirs(os.path.dirname(SNAPSHOT_PATH), exist_ok=True)

def make_duckdb_snapshot(con, dest_path: str):
    # Try to detach if already attached
    try:
        con.execute("DETACH snapshot;")
    except Exception:
        pass  # ignore if not attached

    # Attach destination DB (created if missing)
    con.execute(f"ATTACH '{dest_path}' AS snapshot;")

    try:
        # List base tables in the current (working) DB
        tables = con.execute("""
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE table_type = 'BASE TABLE'
              AND table_schema NOT IN ('information_schema')
            ORDER BY table_schema, table_name
        """).fetchall()

        # Copy each table into snapshot.main
        for schema, table in tables:
            src = f'"{schema}"."{table}"'
            dst = f'snapshot.main."{table}"'
            con.execute(f"CREATE OR REPLACE TABLE {dst} AS SELECT * FROM {src}")

        # Quick row counts
        counts = [(tbl, con.execute(f'SELECT COUNT(*) FROM snapshot.main."{tbl}"').fetchone()[0])
                  for _, tbl in tables]
        print("Snapshot tables:", counts)

    finally:
        # Always detach so the file is unlocked
        con.execute("DETACH snapshot;")
        print(f"Snapshot written to {dest_path} (ready to open in DBeaver)")

make_duckdb_snapshot(con, SNAPSHOT_PATH)


Snapshot tables: [('hl7_dft_identifiers', 1206), ('hl7_dft_identifiers', 1206), ('hl7_dft_keys', 2622), ('hl7_dft_keys', 2622), ('hl7_oru_dft_matches', 4466), ('hl7_oru_dft_matches', 4466), ('hl7_oru_identifiers', 2128), ('hl7_oru_identifiers', 2128), ('hl7_oru_keys', 6594), ('hl7_oru_keys', 6594)]
Snapshot written to C:\Data Generator\data\hl7.duckdb (ready to open in DBeaver)
